# Reproduksi SOTANER — versi **Google Colab + HuggingFace + GPU**

*"What do we really know about State of the art NER?"* (Vajjala & Balasubramaniam, LREC 2022).

Versi ini setara isi dengan `SOTANER_Reproduction.ipynb` (Tabel 2–8 + Figure 1, kolom
**Reported on Paper / Obtained / Delta**) tetapi:

- **Dataset dari HuggingFace** — `conll2012_ontonotesv5` config `english_v4` (split shared-task
  CoNLL-2012, test = 11.257 entitas = support Tabel 3 paper). Tidak perlu file LDC lokal.
- **Jalan di Google Colab** dengan **GPU** — seluruh inferensi & training model di GPU;
  prep data (HF→BIO, split, perturbasi) tetap CPU (ringan).
- **Output persist ke Google Drive** — `FULL_RUN` besar bisa lintas sesi; `RESUME=True`
  melanjutkan dari fold/tahap yang sudah selesai bila runtime Colab putus.

**Prasyarat**: upload folder `SOTANER Notebook Based/` (minimal `helpers/` + `config.cfg`)
ke Google Drive di `MyDrive/SOTANER Notebook Based/`. Sesuaikan `PROJECT_DIR` di sel 0.4.

| Section | Isi | Tabel/Figure |
|---|---|---|
| **0** | setup Colab: GPU, install, mount Drive, konfigurasi | — |
| **1** | build BIO dari HF + validasi + Tabel 2 | Tabel 2 |
| **2** | black-box: per tipe / source / genre / adversarial | Tabel 3–6 |
| **3** | training: 10 random split + uji-t; cross-genre | Tabel 7, 8, Figure 1 |

---
# SECTION 0 — Setup Colab

### 0.1 Konfirmasi GPU

In [ ]:
!nvidia-smi

### 0.2 Install dependensi

Versi dipin dekat env lokal (`spacy 3.8`, `stanza 1.14`, `spark-nlp 5.5.3`, `pyspark 3.5`,
`seqeval 1.2.2`). Sel ini ~4–7 menit sekali per runtime.

**Catatan:**
- `seqeval` hanya rilis *source* (43 kB) dan `setup.py`-nya pakai `setuptools_scm` yang
  gagal mendeteksi versi tanpa `.git` di Colab → `metadata-generation-failed`. Solusi:
  set `SETUPTOOLS_SCM_PRETEND_VERSION=1.2.2` saat memasangnya (sudah di sel ini).
- `spark-nlp` butuh `pyspark` + Java; Colab biasanya sudah punya Java 11.
- Bila Colab menampilkan **"RESTART RUNTIME"** setelah sel ini: klik restart, lalu lanjut
  dari **sel 0.3** (jangan ulang 0.2).

In [ ]:
import sys

# 1) build tooling (setuptools <82 supaya tak bentrok dengan torch bawaan Colab)
!pip -q install -U pip "setuptools<82" wheel

# 2) library dengan wheel
!pip -q install "spacy==3.8.*" spacy-transformers "spark-nlp==5.5.3" "pyspark==3.5.4" faker "datasets>=2.19" scipy openpyxl

# 3) CuPy untuk spaCy di GPU (Colab CUDA 12)
!pip -q install cupy-cuda12x

# 4) stanza + seqeval (seqeval: paksa versi supaya setuptools_scm tak perlu deteksi git)
!pip -q install "stanza==1.14.*"
!SETUPTOOLS_SCM_PRETEND_VERSION=1.2.2 pip -q install seqeval

# 5) model
!{sys.executable} -m spacy download en_core_web_trf
!{sys.executable} -m spacy download en_core_web_lg
!{sys.executable} -c "import stanza; stanza.download('en')"

# Java untuk Spark (aktifkan hanya bila perintah 'java' tak ada):
# !apt-get -qq install -y openjdk-11-jdk-headless

print("\n== versi (nama modul -> versi paket) ==")
import importlib
from importlib.metadata import version as _pkgver, PackageNotFoundError
_MODS = {"spacy": "spacy", "spacy_transformers": "spacy-transformers", "stanza": "stanza",
         "sparknlp": "spark-nlp", "pyspark": "pyspark", "seqeval": "seqeval",
         "datasets": "datasets", "scipy": "scipy", "faker": "faker", "cupy": "cupy-cuda12x"}
for mod, pkg in _MODS.items():
    try:
        importlib.import_module(mod)
        try: v = _pkgver(pkg)
        except PackageNotFoundError: v = "OK (terpasang)"
        print(f"  {mod:18s} {v}")
    except Exception as e:
        note = "  -> spaCy akan jalan di CPU" if mod == "cupy" else ""
        print(f"  {mod:18s} TIDAK BISA DI-IMPORT ({e}){note}")

print("\n>>> Jika di atas ada 'Restart to reload dependencies' dari spaCy: "
      "Runtime > Restart session, lalu lanjut dari sel 0.3 (jangan ulang 0.2).")

### 0.3 Mount Google Drive + path project

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# >>> SESUAIKAN bila folder di Drive kamu bernama lain <<<
PROJECT_DIR = "/content/drive/MyDrive/SOTANER Notebook Based"

assert os.path.isdir(os.path.join(PROJECT_DIR, "helpers")), (
    f"folder helpers/ tidak ditemukan di {PROJECT_DIR!r}. "
    "Upload folder 'SOTANER Notebook Based/' (minimal helpers/ + config.cfg) ke Drive dulu.")
assert os.path.isfile(os.path.join(PROJECT_DIR, "config.cfg")), "config.cfg tidak ada di PROJECT_DIR"

sys.path.insert(0, PROJECT_DIR)
CONFIG = os.path.join(PROJECT_DIR, "config.cfg")

# semua output Colab di sub-folder terpisah dari run lokal
WORK_DIR = os.path.join(PROJECT_DIR, "colab_run")
from pathlib import Path
WORK = Path(WORK_DIR)
DATA_DIR       = WORK / "data"
BIO_DIR        = DATA_DIR / "bio"
SPLITS_DIR     = DATA_DIR / "bio-splits"
CROSSGENRE_DIR = BIO_DIR / "crossgenre"
PERTURB_DIR    = BIO_DIR / "perturb"
SPACY_FMT_DIR  = DATA_DIR / "spacy-fmt"
STANZA_FMT_DIR = DATA_DIR / "stanza-fmt"
SPARK_FMT_DIR  = DATA_DIR / "spark-format"
RESULTS_DIR    = WORK / "results"
MODELS_DIR     = WORK / "models"
TMP_DIR        = Path("/content/tmp")          # scratch besar -> local disk, bukan Drive
for d in (DATA_DIR, RESULTS_DIR, MODELS_DIR, TMP_DIR):
    d.mkdir(parents=True, exist_ok=True)
for sub in ("A", "B", "C", "D"):
    (RESULTS_DIR / sub).mkdir(exist_ok=True)
print("PROJECT_DIR :", PROJECT_DIR)
print("WORK_DIR    :", WORK_DIR, "(output persist di Drive)")

### 0.4 Sel konfigurasi global

In [ ]:
import time, json, glob, itertools, importlib
import pandas as pd, numpy as np

import helpers
from helpers import paper_values as PV
from helpers.report_utils import (comparison_table, export_all, f1_of,
                                  micro_f1_from_text, summarise_folds)

# --- run knobs ----------------------------------------------------------------
FULL_RUN     = False    # Section 3: False = 3 fold + step kecil (cek alur); True = 10 fold skala paper
RUN_SPARKNLP = True     # Spark NLP di semua eksperimen (NerDL GPU best-effort)
RESUME       = True     # lewati fold/tahap yang hasilnya sudah ada di Drive
SEED         = 42
HF_CONFIG    = "english_v4"
USE_GPU      = True

N_FOLDS          = 10    if FULL_RUN else 3
SPACY_MAX_STEPS  = 20000 if FULL_RUN else 3000
STANZA_MAX_STEPS = 8000  if FULL_RUN else 3000
STANZA_BATCH     = 32
SPARK_MAX_EPOCHS = 12    if FULL_RUN else 2

GENRES, NEWS, ALL6 = helpers.GENRES, helpers.NEWS, helpers.ALL6
CROSS = helpers.CROSS_GENRES
ENTITY_TYPES = helpers.ENTITY_TYPES
LIBS = ["spacy", "stanza"] + (["sparknlp"] if RUN_SPARKNLP else [])
ALL_TABLES = {}

try:
    import torch; _cuda = torch.cuda.is_available()
except Exception:
    _cuda = False
print(f"FULL_RUN={FULL_RUN}  RUN_SPARKNLP={RUN_SPARKNLP}  RESUME={RESUME}  N_FOLDS={N_FOLDS}")
print(f"CUDA available: {_cuda}   HF_CONFIG={HF_CONFIG}")
if USE_GPU and not _cuda:
    print("PERINGATAN: USE_GPU=True tapi CUDA tidak terdeteksi -> semua akan jalan di CPU. "
          "Runtime > Change runtime type > GPU.")

---
# SECTION 1 — Dataset (HuggingFace) + Tabel 2

`helpers.hf_to_bio.build_all` menarik `conll2012_ontonotesv5/english_v4` dari export parquet
HuggingFace dan menulis tree BIO 4 kolom yang **identik** dengan versi `conll-2012/v4` lokal
(sudah diverifikasi byte-identik token+tag). `pt` (Alkitab) dikecualikan dari `all6`/`everything`.

### 1.1 Build BIO tree dari HuggingFace

In [ ]:
_report_path = BIO_DIR / "_genre_report.txt"
if RESUME and _report_path.exists():
    print("RESUME: BIO tree sudah ada, lewati build.\n")
    print(_report_path.read_text(encoding="utf-8"))
else:
    t0 = time.time()
    helpers.hf_to_bio.build_all(str(BIO_DIR), config=HF_CONFIG)
    print(f"\nselesai {time.time()-t0:.0f} dtk -> {BIO_DIR}")

### 1.2 Validasi (harus cocok paper)

In [ ]:
from helpers.bio_utils import read_bio_file, count_entities, entity_type_counts

rows = []
for split in ["train", "development", "test"]:
    for name in ["news", "all6", *GENRES]:
        p = BIO_DIR / split / f"onto.{name}.ner"
        if p.exists():
            s, t = read_bio_file(str(p))
            rows.append({"split": split, "subset": name, "sentences": len(s),
                         "entities": count_entities(t)})
display(pd.DataFrame(rows).pivot_table(index="subset", columns="split",
        values=["sentences", "entities"], sort=False))

s, t = read_bio_file(str(BIO_DIR / "test" / "onto.all6.ner"))
assert len(s) == 8262 and count_entities(t) == 11257, (len(s), count_entities(t))
print(f"OK  test/onto.all6.ner = {len(s)} kalimat / {count_entities(t)} entitas "
      f"== support micro Tabel 3 paper")
display(pd.Series(dict(sorted(entity_type_counts(t).items(), key=lambda x: -x[1]))))

### 1.3 Tabel 2 — *Performance of OntoNotes NER models in the three NLP libraries*

`Reported on Paper` = kolom *Obtained* Tabel 2 paper (spaCy 89.09 · Stanza 88.71 · Spark NLP 88.60).
`Obtained` = F1 micro entity-level (seqeval) model stok pada `test/onto.all6.ner`.
Semua model di **GPU**.

In [ ]:
from helpers.eval_blackbox import load_spacy, load_stanza, evaluate_bio, run_blackbox

NLP = load_spacy(gpu=USE_GPU)
STZ = load_stanza(use_gpu=USE_GPU)

STD_TEST = str(BIO_DIR / "test" / "onto.all6.ner")
t0 = time.time()
std_report = evaluate_bio(STD_TEST, nlp=NLP, stanza_tagger=STZ)
print(f"eval spaCy + Stanza: {time.time()-t0:.0f} dtk")
with open(RESULTS_DIR / "A" / "onto.all6.txt", "w", encoding="utf-8") as fh:
    for m in ("spacy", "stanza"):
        fh.write(f"Classification report for {m.capitalize()} NER:\n{std_report[m]['text']}\n\n")
obt_t2 = {"spacy": f1_of(std_report["spacy"]), "stanza": f1_of(std_report["stanza"])}
print({k: round(v, 2) for k, v in obt_t2.items()})

In [ ]:
SPARK = SN_PIPELINE = sn_std_report = None
if RUN_SPARKNLP:
    from helpers import eval_sparknlp
    from helpers.spark_session import start_spark, stage_pretrained_models
    # spark-nlp 5.5.3 .pretrained("bert_base_cased") resolves to a DistilBert token
    # classifier -> ClassCastException. Pre-stage the exact 2020 TF editions the paper
    # / SOTANER Windows run used into ~/cache_pretrained/ so load_bert/load_nerdl load
    # them from disk (~400 MB, re-downloaded each Colab session).
    stage_pretrained_models()
    SPARK = start_spark(memory="12g", gpu=USE_GPU)
    print("Spark", SPARK.version)
    SN_PIPELINE = eval_sparknlp.build_pipeline()
    sn_std = eval_sparknlp.evaluate_bio_sparknlp(
        [STD_TEST], spark=SPARK, pipeline=SN_PIPELINE,
        tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="A")
    sn_std_report = sn_std["onto.all6"]
    obt_t2["sparknlp"] = f1_of(sn_std_report)
    print("Spark NLP micro-F1:", round(obt_t2["sparknlp"], 2))

In [ ]:
t2 = pd.DataFrame({
    "Reported on Paper": [PV.TABLE2_OBTAINED[l] for l in LIBS],
    "Obtained":          [round(obt_t2[l], 2) for l in LIBS],
    "Delta":             [round(obt_t2[l] - PV.TABLE2_OBTAINED[l], 2) for l in LIBS],
    "Reported (situs library)": [PV.TABLE2_WEBSITE[l] for l in LIBS],
}, index=[PV.LIB_LABEL[l] for l in LIBS])
t2.index.name = "Library"
ALL_TABLES["Tabel2_library_check"] = t2
display(t2)

---
# SECTION 2 — Black-box Experiments (Tabel 3, 4, 5, 6)

Identik dengan versi lokal. Model spaCy + Stanza dari Section 1 dipakai ulang (di GPU).

### 2.1 Tabel 3 — F-score per tipe entitas (18 tipe)

In [ ]:
obt_t3 = {}
for et in ENTITY_TYPES:
    obt_t3[et] = {"spacy": f1_of(std_report["spacy"], label=et),
                  "stanza": f1_of(std_report["stanza"], label=et)}
    if sn_std_report is not None:
        obt_t3[et]["sparknlp"] = f1_of(sn_std_report, label=et)
t3 = comparison_table("Entity type", ENTITY_TYPES, obt_t3, PV.TABLE3_ALL18, libs=LIBS)
t3.insert(0, "in paper Table 3",
          ["yes" if et in helpers.TABLE3_TYPES else "" for et in ENTITY_TYPES])
ALL_TABLES["Tabel3_per_type"] = t3
display(t3)

### 2.2 Tabel 4 — per *source*

In [ ]:
SRC = ["bn", "mz", "nw", "bc", "tc", "wb"]
src_files = [str(BIO_DIR / "test" / f"onto.{g}.ner") for g in SRC]
src_res, NLP, STZ = run_blackbox(src_files, str(RESULTS_DIR), "A", nlp=NLP, stanza_tagger=STZ)
sn_src = {}
if RUN_SPARKNLP:
    sn_src = eval_sparknlp.evaluate_bio_sparknlp(src_files, spark=SPARK, pipeline=SN_PIPELINE,
             tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="A")
obt_t4 = {}
for g in SRC:
    stem = f"onto.{g}"
    obt_t4[g] = {"spacy": f1_of(src_res[stem]["spacy"]), "stanza": f1_of(src_res[stem]["stanza"])}
    if RUN_SPARKNLP:
        obt_t4[g]["sparknlp"] = f1_of(sn_src[stem])
t4 = comparison_table("Source", SRC, obt_t4, PV.TABLE4_SOURCE, libs=LIBS)
ALL_TABLES["Tabel4_per_source"] = t4
display(t4)

### 2.3 Tabel 5 — per *genre* (News = bn+mz+nw; bc/tc/wb sama dengan Tabel 4)

In [ ]:
news_file = str(BIO_DIR / "test" / "onto.news.ner")
news_res, NLP, STZ = run_blackbox([news_file], str(RESULTS_DIR), "A", nlp=NLP, stanza_tagger=STZ)
sn_news = {}
if RUN_SPARKNLP:
    sn_news = eval_sparknlp.evaluate_bio_sparknlp([news_file], spark=SPARK, pipeline=SN_PIPELINE,
              tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="A")
obt_t5 = {"news": {"spacy": f1_of(news_res["onto.news"]["spacy"]),
                   "stanza": f1_of(news_res["onto.news"]["stanza"])}}
if RUN_SPARKNLP:
    obt_t5["news"]["sparknlp"] = f1_of(sn_news["onto.news"])
for g in ["bc", "tc", "wb"]:
    obt_t5[g] = obt_t4[g]
t5 = comparison_table("Genre", ["news", "bc", "tc", "wb"], obt_t5, PV.TABLE5_GENRE, libs=LIBS)
ALL_TABLES["Tabel5_per_genre"] = t5
display(t5)

### 2.4 Tabel 6 — adversarial test sets

P1 PERSON→"Dodo" · P2 en_US · P3 en_IN · P4 en_TH♀ · P5 en_IN♀ · P6 GPE en_IE (ber-seed).
Tiga tabel: All (micro-F1), PER (P1–P5), GPE (P6).

In [ ]:
from helpers.perturb import make_all_perturbations
pert = make_all_perturbations(STD_TEST, str(PERTURB_DIR), seed=SEED)
pert_files = [pert[f"perturb{i}"] for i in range(1, 7)]
pr_res, NLP, STZ = run_blackbox(pert_files, str(RESULTS_DIR), "B", nlp=NLP, stanza_tagger=STZ)
sn_pert = {}
if RUN_SPARKNLP:
    sn_pert = eval_sparknlp.evaluate_bio_sparknlp(pert_files, spark=SPARK, pipeline=SN_PIPELINE,
              tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="B")

def pf1(stem, lib, label):
    return f1_of(sn_pert[stem], label=label) if lib == "sparknlp" else f1_of(pr_res[stem][lib], label=label)
def base_f1(lib, label):
    return f1_of(sn_std_report if lib == "sparknlp" else std_report[lib], label=label)

rows_all, rows_per = {}, {}
rows_all["None"] = {l: base_f1(l, "micro avg") for l in LIBS}
rows_per["None"] = {l: base_f1(l, "PERSON") for l in LIBS}
for i in range(1, 6):
    stem = f"onto.all.test.perturb{i}"
    rows_all[f"P{i}"] = {l: pf1(stem, l, "micro avg") for l in LIBS}
    rows_per[f"P{i}"] = {l: pf1(stem, l, "PERSON") for l in LIBS}
rows_all["P6"] = {l: pf1("onto.all.test.perturb6", l, "micro avg") for l in LIBS}
rows_gpe = {"None": {l: base_f1(l, "GPE") for l in LIBS},
            "P6":   {l: pf1("onto.all.test.perturb6", l, "GPE") for l in LIBS}}

t6_all = comparison_table("Setting", ["None","P1","P2","P3","P4","P5","P6"], rows_all, PV.TABLE6_ALL, libs=LIBS)
t6_all.insert(0, "perturbation", ["(baseline)"] + [PV.PERTURB_DEFS[f"P{i}"] for i in range(1, 7)])
t6_per = comparison_table("Setting (F1 kelas PERSON)", ["None","P1","P2","P3","P4","P5"], rows_per, PV.TABLE6_CLASS, libs=LIBS)
t6_gpe = comparison_table("Setting (F1 kelas GPE)", ["None","P6"], rows_gpe,
                          {"None": PV.TABLE6_NONE_GPE, "P6": PV.TABLE6_CLASS["P6"]}, libs=LIBS)
ALL_TABLES["Tabel6_All"] = t6_all; ALL_TABLES["Tabel6_PER"] = t6_per; ALL_TABLES["Tabel6_GPE"] = t6_gpe
display(t6_all); display(t6_per); display(t6_gpe)

---
# SECTION 3 — Training NER Experiments (Tabel 7, 8, Figure 1) — **GPU**

Semua training di GPU: spaCy `--gpu-id 0`, Stanza `use_gpu=True`, Spark NLP sesi `gpu=True`
(NerDL best-effort). `RESUME=True` → fold yang hasilnya sudah ada di Drive dilewati.

| | `FULL_RUN=False` | `FULL_RUN=True` |
|---|---|---|
| fold | 3 | 10 |
| spaCy max_steps | 3 000 | 20 000 |
| Stanza max_steps | 3 000 | 8 000 |
| Spark maxEpochs | 2 | 12 |

### 3.1 Buat 10 random split (seeded)

In [ ]:
from helpers.splits import make_kfold_splits
if RESUME and (SPLITS_DIR / "fold10_test.ner").exists():
    print("RESUME: split sudah ada.")
else:
    make_kfold_splits(str(BIO_DIR / "onto.everything.ner"), str(SPLITS_DIR), n_splits=10, seed=SEED)
print(f"memakai {N_FOLDS} dari 10 fold.")

### 3.2a spaCy folds (GPU)

In [ ]:
from helpers import train_spacy
spacy_fold_f1 = []
for k in range(1, N_FOLDS + 1):
    ej = RESULTS_DIR / "C" / f"spacy_fold{k}.json"
    mb = MODELS_DIR / "spacy-folds" / f"fold{k}" / "model-best"
    if RESUME and ej.exists():
        j = json.load(open(ej)); f1 = (j.get("ents_f") or j.get("ner", {}).get("ents_f", 0)) * 100
        print(f"  spaCy fold{k}: F1 = {f1:.2f}  (dari checkpoint)")
    else:
        tr = train_spacy.prep(str(SPLITS_DIR / f"fold{k}_train.ner"), str(SPACY_FMT_DIR))
        dv = train_spacy.prep(str(SPLITS_DIR / f"fold{k}_dev.ner"),   str(SPACY_FMT_DIR))
        te = train_spacy.prep(str(SPLITS_DIR / f"fold{k}_test.ner"),  str(SPACY_FMT_DIR))
        if not (RESUME and mb.is_dir()):
            train_spacy.train(tr, dv, str(mb.parent), max_steps=SPACY_MAX_STEPS, gpu_id=0, config=CONFIG)
        f1 = train_spacy.evaluate(str(mb), te, str(ej))
        print(f"  spaCy fold{k}: F1 = {f1:.2f}")
    spacy_fold_f1.append(f1)
print("spaCy folds:", [round(x, 2) for x in spacy_fold_f1])

### 3.2b Stanza folds (GPU)

In [ ]:
from helpers import train_stanza
stanza_fold_f1 = []
for k in range(1, N_FOLDS + 1):
    rep = RESULTS_DIR / "C" / f"stanza_fold{k}.txt"
    _ck = micro_f1_from_text(rep.read_text(encoding="utf-8")) if (RESUME and rep.exists()) else None
    if _ck is not None:
        f1 = _ck
        print(f"  Stanza fold{k}: F1 = {f1:.2f}  (dari checkpoint)")
    else:
        trj = str(STANZA_FMT_DIR / f"fold{k}_train.json"); train_stanza.bio_to_json(str(SPLITS_DIR / f"fold{k}_train.ner"), trj)
        dvj = str(STANZA_FMT_DIR / f"fold{k}_dev.json");   train_stanza.bio_to_json(str(SPLITS_DIR / f"fold{k}_dev.ner"),   dvj)
        tej = str(STANZA_FMT_DIR / f"fold{k}_test.json");  train_stanza.bio_to_json(str(SPLITS_DIR / f"fold{k}_test.ner"),  tej)
        model = train_stanza.train(trj, dvj, f"en_fold{k}", str(MODELS_DIR / "stanza-folds"),
                                   f"fold{k}.pt", max_steps=STANZA_MAX_STEPS, use_gpu=USE_GPU, batch_size=STANZA_BATCH)
        res = train_stanza.predict_and_score(model, tej, f"en_fold{k}", use_gpu=USE_GPU)
        rep.write_text(res["text"], encoding="utf-8")
        f1 = res["f1"]
        print(f"  Stanza fold{k}: F1 = {f1:.2f}")
    stanza_fold_f1.append(f1)
print("Stanza folds:", [round(x, 2) for x in stanza_fold_f1])

### 3.2c Spark NLP folds (NerDL, GPU best-effort)

In [ ]:
spark_fold_f1 = []
if RUN_SPARKNLP:
    from helpers import train_sparknlp
    for k in range(1, N_FOLDS + 1):
        rep = RESULTS_DIR / "C" / f"spark_fold{k}.txt"
        _ck = micro_f1_from_text(rep.read_text(encoding="utf-8")) if (RESUME and rep.exists()) else None
        if _ck is not None:
            f1 = _ck
            print(f"  Spark NLP fold{k}: F1 = {f1:.2f}  (dari checkpoint)")
        else:
            res = train_sparknlp.run_fold(
                str(SPLITS_DIR / f"fold{k}_train.ner"), str(SPLITS_DIR / f"fold{k}_dev.ner"),
                str(SPLITS_DIR / f"fold{k}_test.ner"), tmp_dir=str(TMP_DIR), spark=SPARK,
                max_epochs=SPARK_MAX_EPOCHS, seed=0)
            rep.write_text(res["text"], encoding="utf-8"); f1 = res["f1"]
            print(f"  Spark NLP fold{k}: F1 = {f1:.2f}")
        spark_fold_f1.append(f1)
    print("Spark NLP folds:", [round(x, 2) for x in spark_fold_f1])
else:
    print("dilewati (RUN_SPARKNLP=False)")

### 3.2d Rakit Tabel 7

In [ ]:
fold_f1 = {"spacy": spacy_fold_f1, "stanza": stanza_fold_f1}
if RUN_SPARKNLP and spark_fold_f1:
    fold_f1["sparknlp"] = spark_fold_f1
t7 = summarise_folds(fold_f1)
t7["paper avg"]  = [PV.TABLE7_SUMMARY[l]["avg"]  for l in fold_f1]
t7["paper sdev"] = [PV.TABLE7_SUMMARY[l]["sdev"] for l in fold_f1]
t7["delta avg"]  = (t7["avg"] - t7["paper avg"]).round(2)
ALL_TABLES["Tabel7_random_splits"] = t7
display(t7)
if not FULL_RUN:
    print("CATATAN: FULL_RUN=False -> cek alur, bukan reproduksi Tabel 7.")

### 3.3 Uji-t berpasangan antar library (paper §5.1)

In [ ]:
from scipy.stats import ttest_rel
rows = []
for a, b in itertools.combinations(fold_f1, 2):
    va, vb = fold_f1[a], fold_f1[b]
    if len(va) > 1:
        stat, p = ttest_rel(va, vb)
        rows.append({"pair": f"{PV.LIB_LABEL[a]} vs {PV.LIB_LABEL[b]}",
                     "mean diff": round(np.mean(va) - np.mean(vb), 3),
                     "t": round(float(stat), 3), "p-value": round(float(p), 4),
                     "signif (p<0.01)": "ya" if p < 0.01 else "tidak"})
tt = pd.DataFrame(rows); ALL_TABLES["Tabel7_ttest"] = tt
display(tt)
if N_FOLDS < 10:
    print(f"CATATAN: N_FOLDS={N_FOLDS} (<10) -> uji-t representatif penuh hanya dengan FULL_RUN=True.")
print("klaim paper: spaCy ~ Spark NLP (tidak beda signifikan), keduanya > Stanza (p<0.01)")

### 3.4 Set cross-genre

In [ ]:
from helpers.crossgenre import make_sets
make_sets(str(BIO_DIR), str(CROSSGENRE_DIR))
TEST_MAP = {g: str(CROSSGENRE_DIR / f"{g}_test.ner") for g in CROSS}

### 3.5–3.6 Helper training cross-genre (latih sekali, uji 4 genre) — GPU

In [ ]:
from helpers import train_spacy, train_stanza
if RUN_SPARKNLP:
    from helpers import train_sparknlp

def cg_spacy(name, train_ner, dev_ner):
    mb = train_spacy.train(train_spacy.prep(train_ner, str(SPACY_FMT_DIR)),
                           train_spacy.prep(dev_ner, str(SPACY_FMT_DIR)),
                           str(MODELS_DIR / "cg" / f"spacy_{name}"),
                           max_steps=SPACY_MAX_STEPS, gpu_id=0, config=CONFIG)
    return {g: train_spacy.evaluate(mb, train_spacy.prep(p, str(SPACY_FMT_DIR)),
            str(RESULTS_DIR / "D" / f"spacy_{name}_{g}.json")) for g, p in TEST_MAP.items()}

def cg_stanza(name, train_ner, dev_ner):
    trj = str(STANZA_FMT_DIR / f"cg_{name}_train.json"); train_stanza.bio_to_json(train_ner, trj)
    dvj = str(STANZA_FMT_DIR / f"cg_{name}_dev.json");   train_stanza.bio_to_json(dev_ner, dvj)
    model = train_stanza.train(trj, dvj, f"en_{name}", str(MODELS_DIR / "cg"),
                               f"stanza_{name}.pt", max_steps=STANZA_MAX_STEPS,
                               use_gpu=USE_GPU, batch_size=STANZA_BATCH)
    out = {}
    for g, p in TEST_MAP.items():
        tej = str(STANZA_FMT_DIR / f"cg_{name}_test_{g}.json"); train_stanza.bio_to_json(p, tej)
        out[g] = train_stanza.predict_and_score(model, tej, f"en_{name}", use_gpu=USE_GPU)["f1"]
    return out

def cg_spark(name, train_ner, dev_ner):
    if not RUN_SPARKNLP:
        return {}
    model, _, bert = train_sparknlp.fit_nerdl(train_ner, str(TMP_DIR), spark=SPARK,
                                              bert=None, max_epochs=SPARK_MAX_EPOCHS)
    return {g: train_sparknlp.score_nerdl(model, p, str(TMP_DIR), SPARK, bert)["f1"]
            for g, p in TEST_MAP.items()}

def run_cg(name, train_ner, dev_ner):
    res = {"spacy": cg_spacy(name, train_ner, dev_ner),
           "stanza": cg_stanza(name, train_ner, dev_ner)}
    if RUN_SPARKNLP:
        res["sparknlp"] = cg_spark(name, train_ner, dev_ner)
    return res

### 3.5 Tabel 8 — latih hanya pada `news`, uji ke 4 genre

In [ ]:
t8_res = run_cg("t8_news", str(CROSSGENRE_DIR / "news_train.ner"), str(CROSSGENRE_DIR / "news_dev.ner"))
obt_t8 = {g: {l: t8_res[l][g] for l in LIBS} for g in CROSS}
t8 = comparison_table("Test genre", CROSS, obt_t8, PV.TABLE8, libs=LIBS)
ALL_TABLES["Tabel8_single_genre"] = t8
display(t8)

### 3.6 Figure 1 — leave-one-genre-out (train 3 genre, dev genre ke-4)

In [ ]:
HELD_TO_TRAIN = {"news": "wb+tc+bc", "bc": "news+wb+tc", "tc": "news+bc+wb", "wb": "news+tc+bc"}
fig1_obt = {held: run_cg(f"f1_not_{held}",
                         str(CROSSGENRE_DIR / f"train_not_{held}.ner"),
                         str(CROSSGENRE_DIR / f"dev_{held}.ner")) for held in CROSS}
recs = []
for held in CROSS:
    for test_g in CROSS:
        for l in LIBS:
            recs.append({"held_out (dev)": held, "trained_on": HELD_TO_TRAIN[held],
                         "test_genre": test_g, "library": PV.LIB_LABEL[l],
                         "Obtained": round(fig1_obt[held][l][test_g], 2),
                         "Reported on Paper": PV.FIGURE1[held][test_g].get(l)})
fig1_df = pd.DataFrame(recs)
fig1_df["Delta"] = (fig1_df["Obtained"] - fig1_df["Reported on Paper"]).round(2)
ALL_TABLES["Figure1_cross_genre"] = fig1_df
display(fig1_df.head(12))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=True)
colors = {"spacy": "#4C72B0", "stanza": "#DD8452", "sparknlp": "#55A868"}
x = np.arange(len(CROSS)); w = 0.8 / len(LIBS)
for ax, held in zip(axes.flat, CROSS):
    for j, l in enumerate(LIBS):
        pos = x + (j - (len(LIBS) - 1) / 2) * w
        ax.bar(pos, [fig1_obt[held][l][g] for g in CROSS], w, label=PV.LIB_LABEL[l], color=colors[l])
        ax.scatter(pos, [PV.FIGURE1[held][g].get(l) for g in CROSS], color="black", s=18, zorder=3)
    ax.set_title(f"train: {HELD_TO_TRAIN[held]}  /  dev: {held}")
    ax.set_xticks(x); ax.set_xticklabels(CROSS); ax.set_ylim(40, 95); ax.grid(axis="y", alpha=.3)
axes.flat[0].legend(loc="lower left", fontsize=9)
fig.suptitle("Figure 1 — Cross-genre (bar = Obtained, • = Reported on Paper)", fontsize=12)
fig.supxlabel("test genre"); fig.supylabel("F1"); fig.tight_layout()
fig.savefig(RESULTS_DIR / "Figure1_cross_genre.png", dpi=130)
plt.show()

### 3.7 Ekspor semua tabel (ke Drive) + download

In [ ]:
path = export_all(ALL_TABLES, str(RESULTS_DIR))
print("tersimpan di Drive:", path)
print("Sheet:", list(ALL_TABLES))
if RUN_SPARKNLP and SPARK is not None:
    SPARK.stop(); print("Spark dihentikan.")
try:
    from google.colab import files
    files.download(path)
except Exception as e:
    print("(lewati download otomatis:", e, ")")

---
## Deviasi dari paper

1. **Data** — HF parquet `conll2012_ontonotesv5` config `english_v4` (loader `.py` ditolak
   `datasets>=4` → revisi `refs/convert/parquet`). Diverifikasi **byte-identik** (token+tag)
   dengan `conll-2012/v4` LDC lokal; test = 11.257 entitas = support Tabel 3 paper.
2. **Device** — inferensi & training model di **GPU Colab**; prep data (HF→BIO, split,
   perturbasi) CPU (ringan). **Spark NLP NerDL GPU best-effort** — bisa jatuh ke CPU bila
   build TF/CUDA tak cocok; cek `nvidia-smi` saat sel training jalan.
3. **spaCy retraining** (Tabel 7/8/Fig 1) = `config.cfg` rilisan penulis (CNN tok2vec
   transition-based parser), bukan fine-tune `en_core_web_trf`. Eval off-the-shelf tetap `en_core_web_trf`.
4. **Stanza** 1.14 (`ner_tagger`, pretrain `conll17.pt`); paper ~1.4 + `combined.pt`.
5. **Spark NLP** 5.5.3 + model `onto_bert_base_cased`/`bert_base_cased` (era 2020, sesuai paper);
   paper menyebut 3.1.2. `NerDLApproach.maxEpochs` tak disebut paper → 2 (cek) / 12 (FULL_RUN).
   Di Colab kedua model di-*stage* dari edisi TF offline JSL S3 (`bert_base_cased` 2.6.0 +
   `onto_bert_base_cased` 2.7.0) ke `/root/cache_pretrained/` via
   `helpers.spark_session.stage_pretrained_models()` — sama dengan run `SOTANER Windows`,
   karena `.pretrained("bert_base_cased")` di spark-nlp 5.5.3 keliru resolve ke
   `DistilBertForTokenClassification` (→ `ClassCastException`).
6. **Tabel 2 "Reported on Paper"** = kolom *Obtained* paper; angka situs library = kolom catatan.
7. **Random split** = `KFold(10, shuffle, random_state=42)` + dev 10% — tafsiran reproducible.
8. **Output** persist di `MyDrive/SOTANER Notebook Based/colab_run/`; `RESUME=True` lanjut dari checkpoint.